<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/08_sentiment_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation of Fine-tuned Sentiment Analysis Model

In this final notebook, we evaluate our fine-tuned **Gemma 3 1B** model. To ensure a fair assessment of its real-world utility, we **filter the test set to only include original headlines**, excluding the synthetic augmentations used during training.

### Evaluation Dimensions
1. **Sentiment Accuracy**: How well does the model predict the human-annotated label?
2. **Tag Integrity**: Does the model reliably follow the `<sentiment>` and `<reasoning>` output format?
3. **Reasoning Quality**: We use **LLM-as-a-Judge** (Qwen 2.5 7B) to grade the model's explanations on a 1-10 scale.

In [ ]:
%%capture
import os
if "COLAB_" in "".join(os.environ.keys()):
    !pip install -U transformers trl accelerate bitsandbytes

In [ ]:
import pandas as pd
import torch
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Gemma3ForCausalLM
from peft import PeftModel, PeftConfig
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, classification_report
import warnings
import gc

warnings.filterwarnings("ignore")

## 1. Loading Data and Model

We load the explained dataset and filter for `is_augmented == False` to ensure we evaluate on high-quality, human-curated headlines.

In [ ]:
DATASET_PATH = "FinancialPhraseBank_explained"
MODEL_PATH = "gemma-3-1b-sentiment-lora"

ds = load_from_disk(DATASET_PATH)
test_df = ds["test"].to_pandas()

# ONLY evaluate on original examples
test_df = test_df[test_df["is_augmented"] == False].reset_index(drop=True)
print(f"Evaluating on {len(test_df)} original test examples.")

In [ ]:
peft_config = PeftConfig.from_pretrained(MODEL_PATH)
BASE_MODEL_ID = peft_config.base_model_name_or_path
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = Gemma3ForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
    ),
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH).eval()

## 2. Evaluation Loop

In [ ]:
INSTRUCTION = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)

def extract_tag(text, tag):
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

results = []
batch_size = 16

for start in tqdm(range(0, len(test_df), batch_size), desc="Evaluating"):
    batch_df = test_df.iloc[start : start + batch_size]
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": INSTRUCTION + f'Headline: "{row["sentence"]}"'}], 
            tokenize=False, add_generation_prompt=True
        ) for _, row in batch_df.iterrows()
    ]
    
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    
    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    
    for i, raw_output in enumerate(decoded):
        sentiment_tag = extract_tag(raw_output, "sentiment")
        reasoning_pred = extract_tag(raw_output, "reasoning")
        results.append({
            "sentiment_true": batch_df.iloc[i]["sentiment"],
            "sentiment_pred": sentiment_tag.lower() if sentiment_tag else "none",
            "reasoning_pred": reasoning_pred if reasoning_pred else "none",
            "tag_integrity": sentiment_tag is not None and reasoning_pred is not None
        })

results_df = pd.DataFrame(results)

## 3. Results Analysis

In [ ]:
print(f"Accuracy: {accuracy_score(results_df['sentiment_true'], results_df['sentiment_pred']):.2%}")
print(f"Tag Integrity: {results_df['tag_integrity'].mean():.2%}")
print("\nClassification Report:")
print(classification_report(results_df["sentiment_true"], results_df["sentiment_pred"]))